In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

In [190]:

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

In [191]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [192]:
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model = model.to(device)

print("Model parameters (total):", sum(p.numel() for p in model.parameters()))

Model parameters (total): 134515008


In [193]:
ALL_WORDS = [
    "idea", "glow", "rust", "maze", "echo", "wisp", "veto", "lush", "gaze", "knit", "fume", "plow",
    "void", "oath", "grim", "crisp", "lunar", "fable", "quest", "verge", "brawn", "elude", "aisle",
    "ember", "crave", "ivory", "mirth", "knack", "wryly", "onset", "mosaic", "velvet", "sphinx",
    "radius", "summit", "banner", "cipher", "glisten", "mantle", "scarab", "expose", "fathom",
    "tavern", "fusion", "relish", "lantern", "enchant", "torrent", "capture", "orchard", "eclipse",
    "frescos", "triumph", "absolve", "gossipy", "prelude", "whistle", "resolve", "zealous",
    "mirage", "aperture", "sapphire",
]

In [194]:
def generate_records():
    for word in ALL_WORDS:
        yield {
            "prompt": (
                f"You spell words with hyphens between the letters like this W-O-R-D.\nWord:\n{word}\n\n"
                + "Spelling:\n"
            ),
            "completion": "-".join(word).upper() + ".",  # Of the form W-O-R-D.
        }


ds = Dataset.from_generator(generate_records)
ds[0]

{'prompt': 'You spell words with hyphens between the letters like this W-O-R-D.\nWord:\nidea\n\nSpelling:\n',
 'completion': 'I-D-E-A.'}

In [195]:
ds = ds.train_test_split(test_size=0.25, seed=42)

In [196]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 46
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 16
    })
})

In [197]:
train_ds = ds["train"]
test_ds = ds["test"]

print(len(train_ds))
print(len(test_ds))

46
16


# base 

In [198]:
def check_spelling(
    model, tokenizer, prompt: str, actual_spelling: str, max_new_tokens: int = 20
):
    inputs = tokenizer(prompt,return_tensors="pt").to(device)
    gen_out = model.generate(**inputs,max_new_tokens=max_new_tokens,use_cache=False)
    decoded_out = tokenizer.decode(gen_out[0],skip_special_tokens=True)
    proposed_spelling = decoded_out.split("Spelling:")[-1].strip().split("\n")[0].strip()
    actual_spelling = actual_spelling.strip()
    chars_correct = sum(1 for a, b in zip(actual_spelling, proposed_spelling) if a == b)
    num_correct = 1 if actual_spelling == proposed_spelling else 0

    print(
        f"Proposed: {proposed_spelling} | Actual: {actual_spelling} "
        f"| Matches: {'YES' if proposed_spelling == actual_spelling else 'NO'}"
    )

    return chars_correct / len(actual_spelling) , num_correct


In [199]:
print(ds["test"][0]["completion"])
check_spelling(
    model=model,
    tokenizer=tokenizer,
    prompt=ds["test"][0]["prompt"],
    actual_spelling=ds["test"][0]["completion"],
)

W-R-Y-L-Y.
Proposed: wry | Actual: W-R-Y-L-Y. | Matches: NO


(0.0, 0)

In [200]:
print(ds["train"][0])

{'prompt': 'You spell words with hyphens between the letters like this W-O-R-D.\nWord:\nsphinx\n\nSpelling:\n', 'completion': 'S-P-H-I-N-X.'}


# Base model Accuracy

In [201]:
correct = 0.0
correct_chars = 0.0
for example in ds["train"]:
    prompt = example["prompt"]
    completion = example["completion"]
    result = check_spelling(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        actual_spelling=completion,
        max_new_tokens=20,
    )
    correct += result[1]
    correct_chars+= result[0]

print(f"{correct}/{len(ds['train'])} words correct\n {correct_chars}/{len(ds['train'])} characters in words correct")

Proposed: sphinx | Actual: S-P-H-I-N-X. | Matches: NO
Proposed: brawn | Actual: B-R-A-W-N. | Matches: NO
Proposed: goss | Actual: G-O-S-S-I-P-Y. | Matches: NO
Proposed: enchant | Actual: E-N-C-H-A-N-T. | Matches: NO
Proposed: tavern | Actual: T-A-V-E-R-N. | Matches: NO
Proposed: whistle | Actual: W-H-I-S-T-L-E. | Matches: NO
Proposed: W-O-R-D | Actual: C-A-P-T-U-R-E. | Matches: NO
Proposed: echo | Actual: E-C-H-O. | Matches: NO
Proposed: mirth | Actual: M-I-R-T-H. | Matches: NO
Proposed: cris | Actual: C-R-I-S-P. | Matches: NO
Proposed: zeal | Actual: Z-E-A-L-O-U-S. | Matches: NO
Proposed:  | Actual: E-M-B-E-R. | Matches: NO
Proposed: scarab | Actual: S-C-A-R-A-B. | Matches: NO
Proposed:  | Actual: K-N-I-T. | Matches: NO
Proposed: resolve | Actual: R-E-S-O-L-V-E. | Matches: NO
Proposed: velvet | Actual: V-E-L-V-E-T. | Matches: NO
Proposed:  | Actual: A-B-S-O-L-V-E. | Matches: NO
Proposed: lunar | Actual: L-U-N-A-R. | Matches: NO
Proposed: maze | Actual: M-A-Z-E. | Matches: NO
Proposed:

# Lets try SFT with LORA

In [202]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params BEFORE: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)


Trainable params BEFORE: 134,515,008 / 134,515,008 (100.00%)


In [203]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=8,
    lora_dropout=0.4,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

In [204]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(
    f"Trainable params AFTER: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)"
)

Trainable params AFTER: 3,686,400 / 138,201,408 (2.67%)


In [205]:
?SFTConfig

Init signature:
SFTConfig(
    output_dir: Optional[str] = None,
    overwrite_output_dir: bool = False,
    do_train: bool = False,
    do_eval: bool = False,
    do_predict: bool = False,
    eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no',
    prediction_loss_only: bool = False,
    per_device_train_batch_size: int = 8,
    per_device_eval_batch_size: int = 8,
    per_gpu_train_batch_size: Optional[int] = None,
    per_gpu_eval_batch_size: Optional[int] = None,
    gradient_accumulation_steps: int = 1,
    eval_accumulation_steps: Optional[int] = None,
    eval_delay: Optional[float] = 0,
    torch_empty_cache_steps: Optional[int] = None,
    learning_rate: float = 2e-05,
    weight_decay: float = 0.0,
    adam_beta1: float = 0.9,
    adam_beta2: float = 0.999,
    adam_epsilon: float = 1e-08,
    max_grad_norm: float = 1.0,
    num_train_epochs: float = 3.0,
    max_steps: int = -1,
    lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType,

In [206]:
from transformers.trainer_utils import SchedulerType
print(SchedulerType._member_names_)

['LINEAR', 'COSINE', 'COSINE_WITH_RESTARTS', 'POLYNOMIAL', 'CONSTANT', 'CONSTANT_WITH_WARMUP', 'INVERSE_SQRT', 'REDUCE_ON_PLATEAU', 'COSINE_WITH_MIN_LR', 'WARMUP_STABLE_DECAY']


In [224]:
output_dir = "tensorboard"
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=100,
    learning_rate=5e-5,
    logging_steps=8,
    eval_strategy="steps",
    eval_steps=8,
    save_strategy="no",
    fp16=False,
    lr_scheduler_type="reduce_lr_on_plateau",
)

In [225]:
trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    args=training_args,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [226]:
trainer.train()

Step,Training Loss,Validation Loss
8,1.234200,1.196498
16,1.181100,1.149626
24,1.129000,1.106094
32,1.083300,1.065544
40,1.023200,1.027261
48,0.996200,0.990844
56,0.956900,0.956151
64,0.888300,0.922978
72,0.881200,0.891799
80,0.843200,0.862753


TrainOutput(global_step=300, training_loss=0.6888021926085154, metrics={'train_runtime': 141.7332, 'train_samples_per_second': 32.455, 'train_steps_per_second': 2.117, 'total_flos': 141777694414080.0, 'train_loss': 0.6888021926085154})

Check Accuracy with SFT

In [227]:
def test(ds_tag='train'):
    correct = 0.0
    correct_chars = 0.0
    for example in ds[ds_tag]:
        prompt = example["prompt"]
        completion = example["completion"]
        result = check_spelling(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            actual_spelling=completion,
            max_new_tokens=20,
        )
        correct += result[1]
        correct_chars+= result[0]

    print(f"{correct}/{len(ds['train'])} words correct\n {correct_chars}/{len(ds['train'])} characters in words correct")

In [228]:
test('train')

Proposed: S-P-H-I-N-K-I-A. | Actual: S-P-H-I-N-X. | Matches: NO
Proposed: B-R-A-N-Y. | Actual: B-R-A-W-N. | Matches: NO
Proposed: G-O-S-H-O-P-I-Y. | Actual: G-O-S-S-I-P-Y. | Matches: NO
Proposed: E-N-C-H-A-N-T. | Actual: E-N-C-H-A-N-T. | Matches: YES
Proposed: T-A-A-N-B-E. | Actual: T-A-V-E-R-N. | Matches: NO
Proposed: W-H-I-S-E. | Actual: W-H-I-S-T-L-E. | Matches: NO
Proposed: C-U-P-A-R-I-N. | Actual: C-A-P-T-U-R-E. | Matches: NO
Proposed: E-C-H-O-R-E. | Actual: E-C-H-O. | Matches: NO
Proposed: M-I-T-H. | Actual: M-I-R-T-H. | Matches: NO
Proposed: C-R-I-S-P. | Actual: C-R-I-S-P. | Matches: YES
Proposed: Z-E-A-L-O-S. | Actual: Z-E-A-L-O-U-S. | Matches: NO
Proposed: E-M-U-R-E. | Actual: E-M-B-E-R. | Matches: NO
Proposed: S-A-C-R-A-B. | Actual: S-C-A-R-A-B. | Matches: NO
Proposed: W-I-N-T. | Actual: K-N-I-T. | Matches: NO
Proposed: R-E-S-I-L-E. | Actual: R-E-S-O-L-V-E. | Matches: NO
Proposed: V-E-L-V-T. | Actual: V-E-L-V-E-T. | Matches: NO
Proposed: A-B-E-U-R-E. | Actual: A-B-S-O-L-V-E. 

In [229]:
test('test')

Proposed: W-R-Y-I-L-Y. | Actual: W-R-Y-L-Y. | Matches: NO
Proposed: G-L-I-N-E-S. | Actual: G-L-I-S-T-E-N. | Matches: NO
Proposed: C-A-S-E-L. | Actual: Q-U-E-S-T. | Matches: NO
Proposed: C-E-R-E-V-E. | Actual: C-R-A-V-E. | Matches: NO
Proposed: L-U-S-I-S-H. | Actual: L-U-S-H. | Matches: NO
Proposed: F-A-L-I-C-E. | Actual: F-A-B-L-E. | Matches: NO
Proposed: K-N-A-R-K. | Actual: K-N-A-C-K. | Matches: NO
Proposed: T-I-R-U-M-P-H-I-L-E. | Actual: T-R-I-U-M-P-H. | Matches: NO
Proposed: S-A-P-I-C-R-I-A. | Actual: S-A-P-P-H-I-R-E. | Matches: NO
Proposed: E-X-P-S-E-R. | Actual: E-X-P-O-S-E. | Matches: NO
Proposed: F-S-R-O-C-S. | Actual: F-R-E-S-C-O-S. | Matches: NO
Proposed: W-E-P-I-S-H. | Actual: W-I-S-P. | Matches: NO
Proposed: M-I-R-E-G-A. | Actual: M-I-R-A-G-E. | Matches: NO
Proposed: I-V-O-Y-U-R-E. | Actual: I-V-O-R-Y. | Matches: NO
Proposed: O-N-S-H-E-R-D. | Actual: O-N-S-E-T. | Matches: NO
Proposed: E-L-E-U-A-D. | Actual: E-L-U-D-E. | Matches: NO
0.0/46 words correct
 10.577976190476189/4

# So only 0 out of all words were correct

I will revisit this once I learn a bit more , i think i will checkout HF's transformer course 

# GRPO

In [230]:
def debug_reward(*args, **kwargs):
    print("\n===== REWARD DEBUG =====")

    print("args:", len(args))

    for i, arg in enumerate(args):
        print(f"\narg[{i}] type =", type(arg))

        if isinstance(arg, list) and len(arg) > 0:
            print("first element:", arg[0])
            print(len(arg))
            print(arg)
        

    print("\nkwargs keys:", kwargs.keys())

    for k, v in kwargs.items():
        print(f"\n{k}:")
        print("type =", type(v))

        if isinstance(v, list) and len(v) > 0:
            print("first element =", v[0])
            print(len(v))
            print(v)

    raise Exception("STOP AFTER DEBUG")

In [231]:
def proportion_correct(word, proposed_spelling):
    correct_spelling = "-".join(word).upper()

    score = 0.0

    max_len = max(len(correct_spelling), len(proposed_spelling))
    proposed_spelling_padded = proposed_spelling.ljust(max_len, " ")
    correct_spelling_padded = correct_spelling.ljust(max_len, " ")

    for a, b in zip(correct_spelling_padded, proposed_spelling_padded):
        if a == b:
            score += 1
        else:
            score -= 1

    return score / (
        len(correct_spelling)
    )


In [232]:
assert proportion_correct("hello", "H-E-L-L-O") == 1
assert proportion_correct("hello", "H-E-L-") == 3 / 9
assert proportion_correct("hello", "H-E-L-L-O!") == 8 / 9

In [233]:
import numpy as np

import re

def reward_spelling(prompts, completions, **kwargs):

    rewards = []
    #print(prompts)
    #print(completions)
    
    for prompt, completion in zip(prompts, completions):

        word = (
            prompt
            .split("Word:\n")[1]
            .split("\n\nSpelling:")[0]
            .strip()
        )
        generated = completion.strip()

        reward = proportion_correct(
            word,
            generated
        )

        rewards.append(reward)
    print(f"Spelling Reward mean and std: {np.mean(rewards):.3f} +/- {np.std(rewards):.3f}")
    return rewards

In [234]:
import re
import numpy as np

def reward_response_in_form_of_letter_dash_letter(
    prompts,
    completions,
    **kwargs
):
    """
    Rewards outputs that look like:
    W-O-R-D
    """

    pattern = re.compile(r"^([A-Z]-)*[A-Z]\.?$")

    completion_strings = [
        completion.split("\n")[0].strip().upper()
        for completion in completions
    ]

    rewards = [
        1.0 if pattern.fullmatch(c) else 0.0
        for c in completion_strings
    ]

    print(
        f"Letter-dash-letter rewards mean/std: "
        f"{np.mean(rewards):.3f} +/- {np.std(rewards):.3f}"
    )

    return rewards

In [235]:
assert reward_response_in_form_of_letter_dash_letter(
    completions=[
        "H-E-L-L-O",
        "hello",
        "H-i!",
    ],
    prompts=[]
) == [1, 0, 0]

Letter-dash-letter rewards mean/std: 0.333 +/- 0.471


In [236]:
from trl import GRPOConfig, GRPOTrainer

In [237]:
training_args = GRPOConfig(
    output_dir="data/spelling-grpo",
    max_completion_length=20,
    logging_steps=5,
    learning_rate=1e-5,
    num_train_epochs=20, 
    per_device_train_batch_size=8,
    num_generations=4,  
    lr_scheduler_type="cosine",
    beta=0.0,
)

In [238]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        reward_spelling,
        reward_response_in_form_of_letter_dash_letter
    ],
    args=training_args,
    train_dataset=ds["train"],
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [239]:
trainer.train()

Spelling Reward mean and std: 0.250 +/- 0.203
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000


Step,Training Loss
5,-0.002100
10,-0.027100
15,0.037800
20,0.002200
25,0.039900
30,0.038300
35,0.077000
40,0.025800
45,0.028600
50,0.037800


Spelling Reward mean and std: 0.252 +/- 0.258
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling Reward mean and std: 0.291 +/- 0.265
Letter-dash-letter rewards mean/std: 0.875 +/- 0.331
Spelling Reward mean and std: 0.424 +/- 0.452
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling Reward mean and std: 0.286 +/- 0.286
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling Reward mean and std: 0.024 +/- 0.453
Letter-dash-letter rewards mean/std: 0.750 +/- 0.433
Spelling Reward mean and std: -0.045 +/- 0.187
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling Reward mean and std: 0.131 +/- 0.233
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling Reward mean and std: 0.339 +/- 0.404
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling Reward mean and std: 0.154 +/- 0.350
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling Reward mean and std: 0.412 +/- 0.259
Letter-dash-letter rewards mean/std: 1.000 +/- 0.000
Spelling 

TrainOutput(global_step=460, training_loss=0.0066804468672236675, metrics={'train_runtime': 751.9714, 'train_samples_per_second': 1.223, 'train_steps_per_second': 0.612, 'total_flos': 0.0, 'train_loss': 0.0066804468672236675})

In [242]:
test('train')

Proposed: S-P-H-I-N-X. | Actual: S-P-H-I-N-X. | Matches: YES
Proposed: B-R-A-N-Y. | Actual: B-R-A-W-N. | Matches: NO
Proposed: G-O-S-H-O-P-I-Y. | Actual: G-O-S-S-I-P-Y. | Matches: NO
Proposed: E-N-C-H-A-N-T. | Actual: E-N-C-H-A-N-T. | Matches: YES
Proposed: T-A-A-N-B. | Actual: T-A-V-E-R-N. | Matches: NO
Proposed: W-H-I-S-E. | Actual: W-H-I-S-T-L-E. | Matches: NO
Proposed: C-U-P-A-R-I-N. | Actual: C-A-P-T-U-R-E. | Matches: NO
Proposed: E-C-H-O-R-E. | Actual: E-C-H-O. | Matches: NO
Proposed: M-I-T-H. | Actual: M-I-R-T-H. | Matches: NO
Proposed: C-R-I-S-P. | Actual: C-R-I-S-P. | Matches: YES
Proposed: Z-E-A-L-O-S. | Actual: Z-E-A-L-O-U-S. | Matches: NO
Proposed: E-M-B-U-R. | Actual: E-M-B-E-R. | Matches: NO
Proposed: S-A-C-R-A-B. | Actual: S-C-A-R-A-B. | Matches: NO
Proposed: W-I-N-T. | Actual: K-N-I-T. | Matches: NO
Proposed: R-E-S-I-L-E. | Actual: R-E-S-O-L-V-E. | Matches: NO
Proposed: V-E-L-V-T. | Actual: V-E-L-V-E-T. | Matches: NO
Proposed: A-B-E-U-R-E. | Actual: A-B-S-O-L-V-E. | Mat

In [243]:
test('test')

Proposed: W-E-R-Y-I-L-Y. | Actual: W-R-Y-L-Y. | Matches: NO
Proposed: G-L-I-N-E-S. | Actual: G-L-I-S-T-E-N. | Matches: NO
Proposed: C-A-S-E-L. | Actual: Q-U-E-S-T. | Matches: NO
Proposed: C-E-R-E-A-V-E. | Actual: C-R-A-V-E. | Matches: NO
Proposed: L-U-S-I-N. | Actual: L-U-S-H. | Matches: NO
Proposed: F-A-L-I-C-E. | Actual: F-A-B-L-E. | Matches: NO
Proposed: K-N-A-R-K. | Actual: K-N-A-C-K. | Matches: NO
Proposed: T-I-R-U-M-P-H-I-L-E. | Actual: T-R-I-U-M-P-H. | Matches: NO
Proposed: S-A-P-I-C-R-I-A. | Actual: S-A-P-P-H-I-R-E. | Matches: NO
Proposed: E-X-P-S-T. | Actual: E-X-P-O-S-E. | Matches: NO
Proposed: F-S-R-O-C-S. | Actual: F-R-E-S-C-O-S. | Matches: NO
Proposed: W-I-P-S. | Actual: W-I-S-P. | Matches: NO
Proposed: M-I-R-E-G. | Actual: M-I-R-A-G-E. | Matches: NO
Proposed: I-V-O-R-Y. | Actual: I-V-O-R-Y. | Matches: YES
Proposed: O-N-S-H-E-R-D. | Actual: O-N-S-E-T. | Matches: NO
Proposed: E-L-E-U-A. | Actual: E-L-U-D-E. | Matches: NO
1.0/46 words correct
 10.694642857142854/46 character

## A definitive improvement , however looks like huge overfitting